# 1. start with small idea of RAG - search by keywords matching

In [1]:
# imports
import os
import glob

from dotenv import load_dotenv
import gradio as gr
import functools
from concurrent.futures import ThreadPoolExecutor
import time
import requests, json
from pathlib import Path
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
import ollama
import chromadb
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_ollama.llms import OllamaLLM
from langchain_community.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from sqlalchemy import false

D:\Code\agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables in a file called .env
load_dotenv(override=True)

llm = OllamaLLM(model="gpt-oss:20b")
MODEL = os.getenv("MODEL", "gpt-oss:20b")
DB_NAME = os.getenv("DB_NAME", "vector_db")
print("Ollama:", llm, "| Model:", MODEL, "| DB:", DB_NAME)

Ollama: OllamaLLM
Params: {} | Model: gpt-oss:20b | DB: vector_db


# 2. RAG bigger idea with vector search - optimized version

In [3]:

# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

folders = glob.glob("knowledge-base/*")

# text_loader_kwargs = {'encoding': 'utf-8'}
# Nếu dòng trên không hoạt động, người dùng Windows có thể dùng dòng dưới thay thế
text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print("Total documents loaded:", len(documents))


Total documents loaded: 36


In [4]:
documents[0]

Document(metadata={'source': 'knowledge-base\\schools\\Học Phí.md', 'doc_type': 'schools'}, page_content='# Thông tin về học phí (VHU)\n\n## Thông Tin\n- **Học phí**: liên hệ Trung Tâm Chăm Sóc Người học để biết thêm chi tiết\n\n\n')

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # Slightly smaller chunks for better retrieval
    chunk_overlap=100,  # Reduced overlap for performance
    separators=["\n\n", "\n", ". ", " ", ""]  # Better separation
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 441 chunks


In [6]:
chunks[9]

Document(metadata={'source': 'knowledge-base\\study\\Công Nghệ Sinh Học.md', 'doc_type': 'study'}, page_content='### Ứng dụng CNTT & ngoại ngữ\n- Đại cương về Công nghệ thông tin và Truyền thông (3 tín chỉ)  \n- Tiếng Anh giao tiếp 1–2 (6 tín chỉ)  \n- Tiếng Trung giao tiếp 1–2 (6 tín chỉ)  \n- Tiếng Nhật giao tiếp 1–2 (6 tín chỉ)  \n- Tiếng Hàn giao tiếp 1–2 (6 tín chỉ)\n\n### Khoa học tự nhiên & môi trường\n- Môi trường và con người (3 tín chỉ)  \n- Thống kê ứng dụng (3 tín chỉ)\n\n### Kinh tế, quản lý & quản trị\n- Tinh thần khởi nghiệp (3 tín chỉ)\n\n### Xã hội, nhân văn & đa văn hóa\n- Văn hiến Việt Nam (3 tín chỉ)\n\n### Tố chất cá nhân chung\n- Phương pháp học đại học (3 tín chỉ)  \n- Quản trị sự thay đổi (3 tín chỉ)')

In [7]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Các loại tài liệu đã tìm thấy: {', '.join(doc_types)}")

Các loại tài liệu đã tìm thấy: study, schools


In [8]:
for chunk in chunks:
    if 'Văn Hiến' in chunk.page_content:
        print(chunk)
        print("_________")

page_content='# Đại học Văn Hiến (Van Hien University – VHU)

## Tổng quan
Đại học Văn Hiến là trường đại học tư thục tại TP. Hồ Chí Minh, định hướng ứng dụng – thực hành, chú trọng kết nối doanh nghiệp và đào tạo theo nhu cầu thị trường lao động.

## Thông tin cơ bản
- **Tên tiếng Việt:** Đại học Văn Hiến  
- **Tên tiếng Anh:** Van Hien University (VHU)  
- **Năm thành lập:** 1997
- **Địa chỉ (các cơ sở):** TP. Hồ Chí Minh (nhiều cơ sở/khối nhà học; sinh viên thường học theo phân khoa & học phần)  
- **Website:** vhu.edu.vn  
- **Số sinh viên:** quy mô vài chục nghìn (bậc đại học chính quy là chủ lực)  
- **Định hướng:** ứng dụng – nghề nghiệp; liên kết doanh nghiệp, học kỳ thực tập  
- **Đặc điểm:** đa ngành, chú trọng kỹ năng mềm, ngoại ngữ, thực hành nghề

## Cấu trúc học thuật' metadata={'source': 'knowledge-base\\schools\\Van Hien University.md', 'doc_type': 'schools'}
_________
page_content='**Ngành Piano - Đại học Văn Hiến (VHU)**

**Tổng quan**

Ngành Piano tại Đại học Văn Hiế

In [9]:
# Đưa các đoạn văn bản (chunks) vào Vector Store, liên kết mỗi đoạn với một vector embedding

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b" # đã pull ở bước 0
)
# Nếu bạn muốn sử dụng embeddings miễn phí từ HuggingFace (thay vì OpenAI),
# hãy thay dòng embeddings = OpenAIEmbeddings()
# bằng:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [10]:
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
# Tạo vector store bằng Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,  # Danh sách các đoạn văn bản đã chia nhỏ
    embedding=embeddings,  # Hàm embedding (ví dụ: OpenAI hoặc HuggingFace)
    persist_directory=DB_NAME  # Thư mục lưu trữ database
)
# Kiểm tra số lượng document đã được lưu vào vector store
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 441 documents


In [11]:
collection = vectorstore._collection

# Lấy 1 embedding từ database
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

# Kiểm tra số chiều (số phần tử trong vector)
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


The vectors have 1,024 dimensions


In [12]:
sample_embedding

array([-0.00787171, -0.0198267 , -0.0045506 , ...,  0.05432117,
        0.02227388,  0.00830718])

In [13]:
# Lấy toàn bộ vector, tài liệu và metadata từ collection
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

# Đưa embedding vào mảng numpy
vectors = np.array(result['embeddings'])

# Lưu lại văn bản
documents = result['documents']

# Trích loại tài liệu từ metadata (giả sử có 'doc_type')
doc_types = [metadata['doc_type'] for metadata in result['metadatas']]

# Gán màu sắc tùy theo loại tài liệu
colors = [['blue', 'green'][['schools', 'study'].index(t)] for t in doc_types]

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


In [14]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ scatter 2D
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Biểu đồ 2D Chroma Vector Store',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show(renderer="browser")

In [15]:
model = ChatOllama(
    model=MODEL,
    temperature=0.7,
    reasoning=False,
)

memory = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True)
retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
)


In [16]:
# Test query with performance monitoring
def test_query_performance():
    """Test query with timing"""
    query = "Bạn có thể mô tả ngắn gọn về Ngành Kinh Doanh Thương Mại không?"
    start_time = time.time()
    result = conversation_chain.invoke({"question": query})
    end_time = time.time()

    print(f"Query processed in {end_time - start_time:.2f} seconds")
    print("Answer:", result["answer"])
    if "source_documents" in result:
        print(f"Used {len(result['source_documents'])} source documents")

In [17]:
test_query_performance()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query processed in 25.97 seconds
Answer: Ngành **Kinh Doanh Thương Mại** (mã ngành 7340121) là chương trình đào tạo đại học với mục tiêu trang bị cho sinh viên nền tảng toàn diện về kinh tế, quản trị, tài chính, luật và kỹ năng chuyên môn cần thiết để làm việc trong các doanh nghiệp, tổ chức thương mại, ngân hàng, thị trường tài chính và các lĩnh vực liên quan đến kinh doanh.

**Cấu trúc chương trình:**
- **Kiến thức giáo dục đại cương** (lý luận chính trị & pháp luật, khoa học tự nhiên, môi trường).
- **Kiến thức cơ sở ngành** (kinh tế vi mô, vĩ mô, hành vi tổ chức, thuế, quản trị sự kiện, phân tích báo cáo tài chính, quản trị nhân sự, tài chính‑tiền tệ, phương pháp nghiên cứu, đạo đức kinh doanh, tiếng Anh chuyên ngành, v.v.).
- **Khối kiến thức chuyên ngành** (56 tín chỉ, 40 tín chỉ bắt buộc) bao gồm các học phần chuyên sâu về tài chính doanh nghiệp, ngân hàng, thị trường tài chính, luật kinh doanh và thương mại, thống kê kinh tế, toán tài chính, kinh tế lượng, v.v.

**Học viên sẽ:*

In [18]:
# set up a new conversation memory for the chat
memory = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [19]:
# Wrapping in a function - note that history isn't used, as the memory is in the conversation_chain

def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

In [20]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


D:\Code\agents\.venv\Lib\site-packages\gradio\analytics.py:106: UserWarning:

IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------

